# Regresión lineal múltiple

Estima la humedad del sustrato dentro de una hora a partir de su valor actual, su valor de hace una hora, la temperatura, la humedad del aire y la hora del día, como plantea la propuesta. Sirve para evaluar si predecir aporta algo frente a solo reaccionar a los umbrales.

Si el sustrato no tiene datos o casi no varía, el objetivo pasa a ser la temperatura (plan B de la propuesta). El notebook avisa cuál usó.

**Cómo se evalúa:**
- Se entrena con la primera parte de los datos y se evalúa con la última, sin barajar, porque en la práctica se predice el futuro con el pasado.
- Se compara contra la **persistencia**, es decir, suponer que el valor seguirá igual. En una serie que cambia despacio es difícil de superar: un modelo que no le gana no aporta valor, aunque su R² se vea alto.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

import comun

SERIE = "#2a78d6"
FRANJA = "#f0efec"
TINTA = "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.edgecolor": "#c9c8c3", "axes.labelcolor": TINTA,
    "xtick.color": TINTA, "ytick.color": TINTA, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e5e1",
    "lines.linewidth": 1.5,
})

ruta = comun.ruta_de_datos()
datos = comun.cargar(ruta)
print(f"Archivo: {ruta.name}")
print(f"Nodo: {', '.join(datos['nodo'].unique())}")

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 1. Objetivo y datos

In [ ]:
HORIZONTE_MIN = 60

objetivo = comun.elegir_objetivo(datos)
cal = comun.calidad(datos)
x, y = comun.matriz_de_regresion(datos, objetivo, HORIZONTE_MIN)
x_entrena, x_prueba, y_entrena, y_prueba = comun.dividir_en_el_tiempo(x, y, separacion_min=HORIZONTE_MIN)

display(Markdown(
    f"- **Objetivo:** {comun.NOMBRES[objetivo]} dentro de {HORIZONTE_MIN} minutos\n"
    f"- **Periodo:** {cal.horas / 24:.1f} días\n"
    f"- **Entrenamiento:** {len(x_entrena):,} filas, hasta {x_entrena.index[-1]:%d/%m %H:%M}\n"
    f"- **Evaluación:** {len(x_prueba):,} filas, desde {x_prueba.index[0]:%d/%m %H:%M}"
))
if objetivo != "humedad_sustrato":
    display(Markdown("**Plan B:** el sustrato no tiene datos suficientes o casi no varió, así que se estima la temperatura."))
if cal.horas < 7 * 24:
    display(Markdown(
        f"**Limitación:** con {cal.horas / 24:.1f} días, el modelo vio pocos ciclos de día y noche. "
        "El resultado es una prueba inicial y no se puede generalizar a otras épocas ni a otro invernadero."
    ))

## 2. Ajuste y comparación contra la persistencia

In [ ]:
modelo = make_pipeline(StandardScaler(), LinearRegression()).fit(x_entrena, y_entrena)
predichos = pd.Series(modelo.predict(x_prueba), index=x_prueba.index)
persistencia = x_prueba["actual"]

comparacion = pd.DataFrame(
    {"Regresión": comun.metricas(y_prueba, predichos), "Persistencia": comun.metricas(y_prueba, persistencia)}
).round(3)
comparacion

In [ ]:
fig, eje = plt.subplots(figsize=(10, 3))
eje.plot(y_prueba.index, y_prueba, color="#898781", label="Medido")
eje.plot(predichos.index, predichos, color=SERIE, label="Regresión")
eje.set_ylabel(comun.NOMBRES[objetivo])
eje.legend(frameon=False)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 3. Peso de cada variable

Coeficientes con las variables estandarizadas: cuánto cambia la predicción por cada desviación estándar de la variable, manteniendo las demás fijas. Como varias variables están correlacionadas, hay que leerlos con cuidado.

In [ ]:
coeficientes = pd.Series(modelo[-1].coef_, index=x.columns, name="coeficiente").sort_values(key=abs, ascending=False)
coeficientes.round(3).to_frame()

## 4. Dónde se equivoca más

Los errores más grandes suelen coincidir con eventos que el modelo no puede anticipar, como un riego: comparar estas horas con el registro de riegos.

In [ ]:
error = (y_prueba - predichos).rename("error")
peores = pd.DataFrame({"medido": y_prueba, "predicho": predichos.round(2), "error": error.round(2)})
peores.loc[error.abs().sort_values(ascending=False).index[:5]]

## 5. Conclusión

Texto generado con los resultados de arriba.

In [ ]:
mae_modelo = comparacion.loc["MAE", "Regresión"]
mae_base = comparacion.loc["MAE", "Persistencia"]
r2 = comparacion.loc["R²", "Regresión"]
mejora = 100 * (mae_base - mae_modelo) / mae_base if mae_base else 0.0
if mejora >= 5 and r2 > 0:
    veredicto = f"La regresión reduce el error {mejora:.1f} % frente a la persistencia y explica parte de la variación (R² = {r2:.2f}): en este periodo, predecir aporta valor."
elif mejora >= 5:
    veredicto = f"La regresión reduce el error {mejora:.1f} % frente a la persistencia, pero su R² es {r2:.2f}: predice peor que el promedio del periodo evaluado. No hay evidencia suficiente de que aporte valor; conviene revisar si los errores vienen de eventos que no puede anticipar, como los riegos."
elif mejora > -5:
    veredicto = f"La regresión y la persistencia quedan prácticamente empatadas ({mejora:+.1f} %): en este periodo, el modelo no aporta una mejora clara."
else:
    veredicto = f"La regresión es {-mejora:.1f} % peor que la persistencia: en este periodo no aporta valor."
display(Markdown(
    f"- Error medio de la regresión: **{mae_modelo:.2f}**; de la persistencia: **{mae_base:.2f}** (en las unidades de {comun.NOMBRES[objetivo]}).\n"
    f"- {veredicto}\n"
    f"- Periodo analizado: {cal.horas / 24:.1f} días, evaluado con {len(x_prueba):,} filas."
))